In [2]:
import requests
import pandas as pd
import json

Была попытка использовать апи ключ themoviedb, однако соединение установить не удалось

In [271]:
api_key = "ab8d15f4e5cfb79fb611f1b4228dd225"
headers ={'accept': 'application/json', 'X-API-KEY': api_key}
res = requests.get("https://api.themoviedb.org/3/search/movie", headers = headers).json()

ConnectionError: HTTPSConnectionPool(host='api.themoviedb.org', port=443): Max retries exceeded with url: /3/search/movie (Caused by NewConnectionError("HTTPSConnection(host='api.themoviedb.org', port=443): Failed to establish a new connection: [WinError 10061] Подключение не установлено, т.к. конечный компьютер отверг запрос на подключение"))

С помощью апи ключа кинопоиска собираем следующие данные:

фильмы с 

1) бюджетом до 10 000 000 $

2) сборами от 10 000 000 $

3) рейтингом кинопоиска от 7

In [4]:
api_key = 'QSE1V6Z-NNXMZG5-KSC8Z3F-8GG76V6'
url = 'https://api.kinopoisk.dev/v1.5/movie?limit=250&rating.kp=7-10&budget.value=1-10000000&fees.world.value=10000000-10000000000000000&selectFields=id&selectFields=name&selectFields=alternativeName&selectFields=fees&selectFields=type&selectFields=year&selectFields=description&selectFields=shortDescription&selectFields=movieLength&selectFields=isSeries&selectFields=totalSeriesLength&selectFields=seriesLength&selectFields=ageRating&selectFields=typeNumber&selectFields=names&selectFields=rating&selectFields=votes&selectFields=genres&selectFields=countries&selectFields=releaseYears&selectFields=budget'
headers ={'accept': 'application/json', 'X-API-KEY': api_key}
movies3 = []
page = requests.get(url, headers = headers)
page = page.json()
movies3.extend(page['docs'])
next_cursor = page.get('next')
for i in range (2):
    url2 = f'{url}&next={next_cursor}'
    page2 = requests.get(url2, headers = headers)
    page2 = page2.json()
    movies3.extend(page2['docs'])

    next_cursor = page2.get('next')
    
    

In [5]:
all_movies = pd.DataFrame(movies3)

Приводим данные к красивому виду

In [6]:
all_movies['kp_votes'] = all_movies['votes'].apply(lambda d: d['kp'])
all_movies['kp_rating'] = all_movies['rating'].apply(lambda d: d['kp'])


all_movies['genre'] = all_movies['genres'].fillna('').apply(lambda genres: ', '.join([genre['name'] for genre in genres]))
all_movies['country'] = all_movies['countries'].fillna('').apply(lambda countries: ', '.join([country['name'] for country in countries]))


all_movies['film_budget'] = all_movies['budget'].apply(lambda d: d['value'])
all_movies['budget currency'] = all_movies['budget'].apply(lambda d: d['currency'])

all_movies['film_fees'] = all_movies['fees'].apply(lambda d: d['world']).apply(lambda d: d['value'])
all_movies['fees currency'] = all_movies['fees'].apply(lambda d: d['world']).apply(lambda d: d['currency'])

In [7]:
all_movies

,names,name,fees,year,shortDescription,isSeries,movieLength,budget,id,alternativeName,...,rating,countries,kp_votes,kp_rating,genre,country,film_budget,budget currency,film_fees,fees currency
0,"[{'name': 'Грязные миллионы', 'language': 'RU'...",Грязные миллионы,"{'world': {'value': 11772156, 'currency': '$'}}",2021,"Бывший коп, потеряв накопления, внедряется в к...",False,109,"{'value': 7000000, 'currency': '$'}",1316625,Boiseu,...,"{'kp': 7.123, 'imdb': 6.4, 'filmCritics': 0, '...",[{'name': 'Корея Южная'}],21740,7.123,"боевик, триллер, криминал, детектив",Корея Южная,7000000,$,11772156,$
1,"[{'name': 'Пропавшая без вести', 'language': '...",Пропавшая без вести,"{'world': {'value': 48767848, 'currency': '$'}...",2023,"Дочь расследует исчезновение матери, уехавшей ...",False,111,"{'value': 7000000, 'currency': '$'}",1304162,Missing,...,"{'kp': 7.202, 'imdb': 7.1, 'filmCritics': 6.8,...",[{'name': 'США'}],32655,7.202,"детектив, триллер, драма",США,7000000,$,48767848,$
2,"[{'name': 'Статья 15', 'language': 'RU', 'type...",Статья 15,"{'world': {'value': 13000000, 'currency': '$'}}",2019,NaN,False,130,"{'value': 4000000, 'currency': '$'}",1264211,Article 15,...,"{'kp': 7.133, 'imdb': 8.1, 'filmCritics': 7.6,...",[{'name': 'Индия'}],276,7.133,"драма, детектив, криминал, триллер",Индия,4000000,$,13000000,$
3,"[{'name': 'Земля кочевников', 'language': 'RU'...",Земля кочевников,"{'world': {'value': 39458207, 'currency': '$'}...",2020,Одинокая Ферн кочует по Америке в доме на коле...,False,110,"{'value': 5000000, 'currency': '$'}",1238506,Nomadland,...,"{'kp': 7.215, 'imdb': 7.3, 'filmCritics': 8.8,...","[{'name': 'США'}, {'name': 'Германия'}]",93172,7.215,драма,"США, Германия",5000000,$,39458207,$
4,"[{'name': 'Залог', 'language': 'RU', 'type': '...",Залог,"{'world': {'value': 14193957, 'currency': '$'}}",2020,NaN,False,113,"{'value': 4000000, 'currency': '$'}",1237995,Dambo,...,"{'kp': 7.867, 'imdb': 7.7, 'filmCritics': 0, '...",[{'name': 'Корея Южная'}],1381,7.867,"драма, комедия, семейный",Корея Южная,4000000,$,14193957,$
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
560,"[{'name': 'Мементо', 'language': 'RU', 'type':...",Мементо,"{'world': {'value': 39723096, 'currency': '$'}...",2000,"Леонард ищет убийцу жены, но ему мешает амнези...",False,113,"{'value': 9000000, 'currency': '$'}",335,Memento,...,"{'kp': 7.871, 'imdb': 8.4, 'filmCritics': 8.8,...",[{'name': 'США'}],381382,7.871,"триллер, детектив, драма, криминал",США,9000000,$,39723096,$
561,"[{'name': 'Пролетая над гнездом кукушки', 'lan...",Пролетая над гнездом кукушки,"{'world': {'value': 108981275, 'currency': '$'...",1975,"Джек Николсон — запертый в психушке бунтарь, к...",False,133,"{'value': 3000000, 'currency': '$'}",336,One Flew Over the Cuckoo's Nest,...,"{'kp': 8.508, 'imdb': 8.6, 'filmCritics': 9.3,...",[{'name': 'США'}],384755,8.508,драма,США,3000000,$,108981275,$
562,"[{'name': 'Окно во двор', 'language': 'RU', 't...",Окно во двор,"{'world': {'value': 36775433, 'currency': '$'}...",1954,Фотограф со сломанной ногой наблюдает через об...,False,112,"{'value': 2000000, 'currency': '$'}",337,Rear Window,...,"{'kp': 8.021, 'imdb': 8.5, 'filmCritics': 9.6,...",[{'name': 'США'}],69008,8.021,"триллер, драма, детектив",США,2000000,$,36775433,$
563,"[{'name': '25-й час', 'language': 'RU', 'type'...",25-й час,"{'world': {'value': 23928503, 'currency': '$'}...",2002,Наркоторговец проводит с друзьями последние су...,False,135,"{'value': 5000000, 'currency': '$'}",320,25th Hour,...,"{'kp': 7.543, 'imdb': 7.6, 'filmCritics': 7.6,...",[{'name': 'США'}],46845,7.543,драма,США,5000000,$,23928503,$


In [8]:
movies_final = all_movies[['name','year','genre','country','film_budget','budget currency','film_fees','fees currency','type','kp_votes','kp_rating']]

In [9]:
movies_final

,name,year,genre,country,film_budget,budget currency,film_fees,fees currency,type,kp_votes,kp_rating
0,Грязные миллионы,2021,"боевик, триллер, криминал, детектив",Корея Южная,7000000,$,11772156,$,movie,21740,7.123
1,Пропавшая без вести,2023,"детектив, триллер, драма",США,7000000,$,48767848,$,movie,32655,7.202
2,Статья 15,2019,"драма, детектив, криминал, триллер",Индия,4000000,$,13000000,$,movie,276,7.133
3,Земля кочевников,2020,драма,"США, Германия",5000000,$,39458207,$,movie,93172,7.215
4,Залог,2020,"драма, комедия, семейный",Корея Южная,4000000,$,14193957,$,movie,1381,7.867
...,...,...,...,...,...,...,...,...,...,...,...
560,Мементо,2000,"триллер, детектив, драма, криминал",США,9000000,$,39723096,$,movie,381382,7.871
561,Пролетая над гнездом кукушки,1975,драма,США,3000000,$,108981275,$,movie,384755,8.508
562,Окно во двор,1954,"триллер, драма, детектив",США,2000000,$,36775433,$,movie,69008,8.021
563,25-й час,2002,драма,США,5000000,$,23928503,$,movie,46845,7.543


In [13]:
movies_final.to_csv('kinopoisk_final_data.csv', index=False)